In [ ]:
import re
import numpy as np
import pandas as pd
import requests
from lxml import etree
import time
from openpyxl import load_workbook

data = pd.ExcelFile('test.xlsx')
sheet1_data = data.parse('Sheet1')
name_list = sheet1_data['search_term'].tolist()
title_list = sheet1_data['chemical_formula'].tolist()
url_list = sheet1_data['link'].tolist()
number_list = sheet1_data['index_number'].tolist()
hxm_list = sheet1_data['title'].tolist()

thresholds = [0, 0.01, 0.02, 0.03, 0.04, 0.05]
results = {f'threshold_{int(t*100)}%': [] for t in thresholds}
error_log = []

def calculate_all_parameters(FF1, FF2, h):
    params = {}
    
    try:
        float_FF1 = list(map(float, FF1)) if FF1 else [0]
        float_FF2 = list(map(float, FF2)) if FF2 else [0]
    except:
        float_FF1 = [0]
        float_FF2 = [0]
    
    try:
        max_FF2_val = max(float_FF2)
        max_index = float_FF2.index(max_FF2_val)
        params['BP'] = FF1[max_index] if FF1 else 0
        params['base_peak_intensity'] = max_FF2_val
        
        if len(float_FF1) > 1:
            bp_mz = float_FF1[max_index]
            other_indices = [i for i in range(len(float_FF1)) if i != max_index]
            if other_indices:
                min_diff = min(abs(float_FF1[i] - bp_mz) for i in other_indices)
                params['BPP'] = min_diff
            else:
                params['BPP'] = 0
        else:
            params['BPP'] = 0
    except:
        params.update({'BP': 0, 'base_peak_intensity': 0, 'BPP': 0})
    
    try:
        params['MaxM'] = max(float_FF1)
        max_idx = float_FF1.index(params['MaxM'])
        
        if len(float_FF1) > 1:
            maxm_mz = float_FF1[max_idx]
            other_indices = [i for i in range(len(float_FF1)) if i != max_idx]
            if other_indices:
                min_diff = min(abs(float_FF1[i] - maxm_mz) for i in other_indices)
                params['MaxMP'] = min_diff
            else:
                params['MaxMP'] = 0
        else:
            params['MaxMP'] = 0
            
        params['MinM'] = min(float_FF1)
    except:
        params.update({'MaxM': 0, 'MaxMP': 0, 'MinM': 0})
    
    params['MM'] = np.mean(float_FF1) if float_FF1 else 0
    params['MSD'] = np.std(float_FF1) if float_FF1 else 0
    params['IM'] = np.mean(float_FF2) if float_FF2 else 0
    params['ISD'] = np.std(float_FF2) if float_FF2 else 0
    
    params['c'] = params['base_peak_intensity']/float(h) if h !=0 else 0
    
    return params

start = 1
output_filename = 'crawler_toxic.xlsx'

for index, (url, title, name, hxm, number) in enumerate(zip(url_list[start-1:], 
                                          title_list[start-1:], 
                                          name_list[start-1:],
                                          hxm_list[start-1:],
                                          number_list[start-1:]),
                                      start=start):
    
    print(f'\nProcessing: {index}/{len(url_list)} | Current link:', url)

    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=30, verify=False)
        response.raise_for_status()
        
        parser = etree.HTML(response.text)
        dd1 = parser.xpath('/html/body/div[4]/text()')
        
        try:
            idx = dd1.index(' m/z int. rel.int.')
            h = float(dd1[idx-2])
        except:
            h = 1
            
        def extract_param(keyword, data):
            for item in data:
                if keyword in item:
                    return item.replace(f' {keyword} ', '').split('(')[0].strip()
            return 'N/A'

        AA2 = [re.sub(r'[\n]', '', item) for item in dd1 if item.count('\xa0') == 4]
        raw_FF1, raw_FF2 = [], []
        for item in AA2:
            parts = [p.strip() for p in item.split('\xa0') if p]
            if len(parts) >= 3:
                raw_FF1.append(parts[0])
                raw_FF2.append(parts[2])

        if not raw_FF2:
            raise ValueError("No valid peak data")

        max_int = max(map(float, raw_FF2))
        
        for threshold in thresholds:
            current_threshold = max_int * threshold
            
            filtered = [(f1, f2) for f1, f2 in zip(raw_FF1, raw_FF2) 
                       if float(f2) >= current_threshold]
            FF1 = [x[0] for x in filtered]
            FF2 = [x[1] for x in filtered]
            
            PN = len(FF1)
            try:
                max_FF2 = max(map(float, FF2)) if FF2 else 0
                IDensity = max_FF2 / PN if PN > 0 else 0
            except:
                IDensity = 0
            
            extra_params = calculate_all_parameters(FF1, FF2, h)
            
            smiles_nodes = parser.xpath('/html/body/div[4]/b[14]/following-sibling::text()[1]')
            smiles = smiles_nodes[0].strip() if smiles_nodes else "N/A"
            
            record = {
                'link': url,
                'chemical_formula': title,
                'search_term': name,
                'title': hxm,
                'index_number': number,
                'SMILES': smiles,
                'current_threshold': f'{int(threshold*100)}%',
                'original_peak_count': len(raw_FF1),
                'filtered_peak_count': PN,
                'RETENTION_TIME': extract_param('RETENTION_TIME', dd1),
                'COLLISION_ENERGY': extract_param('COLLISION_ENERGY', dd1),
                'PRECURSOR_TYPE': extract_param('PRECURSOR_TYPE', dd1),
                'ION_TYPE': extract_param('ION_TYPE', dd1),
                'baseline_h': h,
                'PN': PN,
                'ID': IDensity,
                **extra_params
            }
            
            results[f'threshold_{int(threshold*100)}%'].append(record)
            
    except Exception as e:
        error_log.append({
            'link': url, 
            'error_message': str(e),
            'time': pd.Timestamp.now()
        })
        print(f"Error processing {url}: {e}")

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    column_order = [
        'link', 'chemical_formula', 'search_term', 'title', 'index_number', 'SMILES', 
        'original_peak_count', 'filtered_peak_count', 'PN', 'ID',
        'BP', 'BPP', 'MaxM', 'MaxMP', 'MinM',
        'MM', 'MSD', 'IM', 'ISD', 'c',
        'RETENTION_TIME', 'COLLISION_ENERGY', 'PRECURSOR_TYPE','ION_TYPE',
        'current_threshold', 'baseline_h'
    ]
    
    for sheet_name, data in results.items():
        if data:
            df = pd.DataFrame(data)
            for col in column_order:
                if col not in df.columns:
                    df[col] = None
            df[column_order].to_excel(writer, sheet_name=sheet_name, index=False)
    
    if error_log:
        pd.DataFrame(error_log).to_excel(writer, sheet_name='error_log', index=False)

print(f"\nProcessing completed! Results saved to {output_filename}")
print(f"Data statistics by threshold:")
for k,v in results.items():
    print(f"{k}: {len(v)} records")